# Clustering in Scanpy

In [ ]:
# This cell is labelled 'paramters' to work with papermill remotely
# Papermill will overwite the default local plate value below with whatever is passed to 
# the -p flag in the snakerule shell script
# os.system("conda activate eqtl_study") use this locally if using VScode
plate = 'plate1'
plate = globals().get("plate")
print(f"Processing plate: {plate}")

### Set Variables

In [ ]:
# Import custom utility packages, lists and functions
import sys
import os
if os.path.exists('/scratch/'):
    root_dir = '/scratch/c.c1477909/eQTL_study_2025/'
else:
    root_dir = '/Users/darren/Desktop/eQTL_study_2025/'
        
sys.path.append(root_dir + 'workflow/scripts/')

from init_env import *
from anndata_utils import *
from gene_lists import *

# Set variables
resolutions = [0.1]# [0.1, 0.2, 0.3] # This takes a while per res
batch_col = 'sample' 
var_genes = 2000
n_pcs = 20
run_genes_in_smpl_filter = True

### Load and merge data

In [ ]:
# Initialize the environment and get all paths and logger
logger, root_dir, sheets_dir, plate_dir, scanpy_dir = initialize_env(plate)
logger.info("Loading data ...")
adata = load_and_dwnsmpl_data(None, True, scanpy_dir + 'adata_qc_plate1.h5ad', scanpy_dir + 'adata_qc_plate2.h5ad')
print(adata.shape)
print(adata.X[:10, :10].todense())

### Additional filter based on number of genes expressed per sample

In [ ]:
# Plot reads per gene distribution
plot_gene_read_distribution(adata)
plot_stacked_figure(adata, sample_column="sample", color_column='leiden_0.1', barplot=False, recalculate_columns=True)

In [ ]:
# Retain genes with at least min_reads in at least min_samples in the dataset
# Failed on min_reads=10, min_samples=100
#if run_genes_in_smpl_filter is True:
#    for min_reads in [1, 5, 10]:
#        for min_samples in [25, 50, 75, 100]:
#            mask = filter_genes_by_read_count(adata, min_reads=min_reads, min_samples=min_samples, inplace=False, verbose=True)
#            filtered_genes = adata.var.index[mask].tolist()
#            print(f"Filtered genes when min reads={min_reads}, min_samples={min_samples}: {len(filtered_genes)} remain.\n")
min_reads = 10
min_samples = 100
filter_genes_by_read_count(adata, min_reads = min_reads, min_samples = min_samples, inplace = True, verbose = True)
print(f"Filtered genes when min reads = {min_reads}, min_samples = {min_samples}: {adata.shape[1]} remain.\n")

# Recalculate QC metrics after filtering genes
#mt_genes = [gene for gene in adata.var_names if gene.upper().startswith("MT-")]
#print(f"Number of mitochondrial genes: {len(mt_genes)}")
#sc.pp.calculate_qc_metrics(adata, inplace=True, log1p=True)

In [ ]:
# Plot reads per gene distribution
plot_gene_read_distribution(adata)
plot_stacked_figure(adata, sample_column = "sample", color_column = 'leiden_0.1', barplot = False, recalculate_columns = True)

# Initial cell counts

In [ ]:
# Cell counts by sample
print(f"Number of samples: {adata.obs['sample'].nunique()}")
adata.obs['sample'].value_counts()

In [ ]:
# Cell counts by sublibrary
adata.obs['sublibrary'] = [x[1] for x in adata.obs.index.str.split('__s')] 
adata.obs['sublibrary'].value_counts()

In [ ]:
# Cell counts by plate
adata.obs['plate'].value_counts()

In [ ]:
# Save original counts to layer
adata.raw = adata.copy()  # Store as raw for DE analysis
adata.layers["counts"] = adata.X.copy()  # Save raw counts
adata.obs

# Classify sex

In [ ]:
logger.info(f"Pull out sex genes ...")
def classify_sample_sex(adata, output_csv = "sex_assignments.csv"):
    """
    Classify samples by sex using expression of XIST, RPS4Y1, and DDX3Y from adata.raw.
    
    Parameters:
        adata (AnnData): Annotated data matrix with raw counts in `adata.raw`.
        output_csv (str): Path to save the resulting CSV file.
    
    Returns:
        pd.DataFrame: DataFrame with sample ID, predicted sex, and numeric code.
    """
    # Step 1: Access raw counts
    raw_adata = adata.raw.to_adata()
    
    # Step 2: Identify available sex genes
    sex_genes = ['XIST', 'RPS4Y1', 'DDX3Y']
    available_genes = [gene for gene in sex_genes if gene in raw_adata.var_names]
    missing_genes = [gene for gene in sex_genes if gene not in raw_adata.var_names]
    print("Available genes in raw data:", available_genes)
    print("Missing genes in raw data:", missing_genes)
    
    if not available_genes:
        raise ValueError("No sex genes found in adata.raw!")
    
    # Step 3: Normalize and log-transform
    sex_raw = raw_adata[:, available_genes]
    sc.pp.normalize_total(sex_raw, target_sum=1e4)
    sc.pp.log1p(sex_raw)
    
    # Step 4: Aggregate by sample
    sex_adata = sex_raw.to_df()
    sample_sex_exp = sex_adata.groupby(adata.obs['sample']).mean()
    
    # Step 5: Thresholds and classification
    xist_thresh = sample_sex_exp['XIST'].median() + sample_sex_exp['XIST'].std()
    y_thresh = 0.5  # adjustable if needed

    def classify_sex(row):
        if row['XIST'] > xist_thresh and row.get('RPS4Y1', 0) < y_thresh and row.get('DDX3Y', 0) < y_thresh:
            return 'Female'
        elif (row.get('RPS4Y1', 0) > y_thresh or row.get('DDX3Y', 0) > y_thresh) and row['XIST'] < xist_thresh:
            return 'Male'
        else:
            return 'Ambiguous'

    sample_sex_exp['Predicted_Sex'] = sample_sex_exp.apply(classify_sex, axis=1)
    sample_sex_exp['Sex_Numeric'] = sample_sex_exp['Predicted_Sex'].map({'Male': 1, 'Female': 2, 'Ambiguous': 0})

    # Step 6: Output
    output_df = sample_sex_exp[['Predicted_Sex', 'Sex_Numeric']].reset_index()
    output_df.columns = ['Sample_ID', 'Predicted_Sex', 'Sex_Numeric']
    output_df.to_csv(output_csv, index=False)
    print(f"Saved to {output_csv}")

    return output_df

    sex_df = classify_sample_sex(adata, output_csv = scanpy_dir + "sex_assignments.csv")
    sex_df

In [ ]:
from sklearn.decomposition import PCA
def visualize_sex_gene_classification(sample_sex_exp, xist_thresh, y_thresh, sex_genes=None):
    if sex_genes is None:
        sex_genes = ['XIST', 'RPS4Y1', 'DDX3Y']
    
    available_genes = [g for g in sex_genes if g in sample_sex_exp.columns]
    if not available_genes:
        raise ValueError("None of the specified sex genes found in sample_sex_exp.")
    
    # Choose a Y gene to visualize (prefer RPS4Y1, fallback to DDX3Y)
    y_gene = 'RPS4Y1' if 'RPS4Y1' in available_genes else 'DDX3Y' if 'DDX3Y' in available_genes else None
    if y_gene is None:
        raise ValueError("No Y-linked gene found among available genes.")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Scatter plot: XIST vs Y gene
    sns.scatterplot(
        data=sample_sex_exp,
        x='XIST',
        y=y_gene,
        hue='Predicted_Sex',
        palette={'Male': 'blue', 'Female': 'red', 'Ambiguous': 'gray'},
        ax=axes[0]
    )
    axes[0].axvline(xist_thresh, color='red', linestyle='--', label='XIST threshold')
    axes[0].axhline(y_thresh, color='blue', linestyle='--', label=f'{y_gene} threshold')
    axes[0].set_title(f'XIST vs {y_gene}')
    axes[0].legend()

    # 2. Boxplot: all genes by Predicted_Sex
    melted = sample_sex_exp.reset_index()[['Predicted_Sex'] + available_genes].melt(id_vars='Predicted_Sex')
    sns.boxplot(data=melted, x='variable', y='value', hue='Predicted_Sex', ax=axes[1])
    axes[1].set_title('Gene Expression by Predicted Sex')
    axes[1].set_xlabel('Gene')
    axes[1].set_ylabel('Log-normalized expression')
    axes[1].legend(loc='upper right')

    # 3. PCA on available sex genes
    df_pca_input = sample_sex_exp[available_genes].dropna()
    pca = PCA(n_components=2)
    pcs = pca.fit_transform(df_pca_input)
    df_pca = pd.DataFrame(pcs, columns=['PC1', 'PC2'], index=df_pca_input.index)
    df_pca['Predicted_Sex'] = sample_sex_exp.loc[df_pca.index, 'Predicted_Sex']

    sns.scatterplot(data=df_pca, x='PC1', y='PC2', hue='Predicted_Sex', palette='Set1', ax=axes[2])
    axes[2].set_title('PCA of Sex Gene Expression')

    plt.tight_layout()
    plt.show()

In [ ]:
# Aggregate expression per donor/sample
sex_gene_exp = adata[:, sex_genes].X.mean(axis=0)  # Averaging across cells
sex_gene_df = pd.DataFrame(sex_gene_exp, index=sex_genes, columns=["Mean Expression"])
print(sex_gene_df)

### Run Scanpy analysis

In [ ]:
# Normalise
logger.info("Normalising ...")
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
# Get highly variable genes
logger.info("Detecting variable genes ...")
sc.pp.highly_variable_genes(adata, n_top_genes = var_genes, flavor = "seurat_v3", batch_key = batch_col)

adata = adata[:,adata.var.highly_variable]
sc.pp.scale(adata, max_value = 10)
sc.pl.highly_variable_genes(adata)

hvg_genes = sorted(adata.var_names[adata.var['highly_variable']])
with open(scanpy_dir + "highly_variable_genes.txt", "w") as f:
    for gene in hvg_genes:
        f.write(f"{gene}\n")

In [ ]:
# PCA
logger.info("PCA ...")
sc.tl.pca(adata, n_comps = n_pcs, use_highly_variable = True)
sc.pl.pca_variance_ratio(adata, log = True, n_pcs = n_pcs, save='') # scanpy generates the filename automatically

In [ ]:
# Create UMAP and Clustering
logger.info("Clustering and UMAP ...")
sc.pp.neighbors(adata, n_neighbors = 10, n_pcs = n_pcs)
sc.tl.umap(adata)

for res in resolutions:
    sc.tl.leiden(adata, resolution = res, key_added = f'leiden_{res}')

In [ ]:
# Plot UMAP
logger.info("Plot UMAP ...")
fig = create_umap_visualisations(adata, resolutions, leiden_prefix = "leiden", clustering_algorithm = "Leiden")
plt.show() 

In [ ]:
# UMAPs by .obs column
obs_columns = ['sublibrary', 'plate']
plot_umap_grid(adata, obs_columns, grid_size = (2, 2), figsize = (10, 8), save_path = None)

### Run batch correction

In [ ]:
# Store the original PCA and UMAP coordinates
adata.obsm['X_pca_pre_harmony'] = adata.obsm['X_pca']
adata.obsm['X_umap_pre_harmony'] = adata.obsm['X_umap']

# Run Harmony
sce.pp.harmony_integrate(adata, 'plate')

# Set PCA for clustering
adata.obsm['X_pca'] = adata.obsm['X_pca_harmony']
sc.pp.neighbors(adata, n_neighbors = 10, n_pcs = n_pcs)
for res in resolutions:
    sc.tl.leiden(adata, resolution = res, key_added = f'leiden_harmony_{res}')
sc.tl.umap(adata) # Overwites X_umap

# Extract PCs as a DataFrame - need this for eQTL analysis expression covariates
expression_pcs = pd.DataFrame(adata.obsm['X_pca'], index = adata.obs.index, 
                              columns = [f'PC{i+1}' for i in range(n_pcs)])
expression_pcs.to_csv(scanpy_dir + "expression_pcs.csv")

In [ ]:
# Plot PCA before Harmony
#sc.pl.pca_scatter(adata, color='plate', title='PCA Before Harmony')

# Temporarily set the Harmony PCA for plotting
#adata.obsm['X_pca'] = adata.obsm['X_pca_harmony']

# Plot PCA after Harmony
#sc.pl.pca_scatter(adata, color='plate', title='PCA After Harmony')

# Restore the original PCA if needed
#adata.obsm['X_pca'] = adata.obsm['X_pca_pre_harmony']

In [ ]:
# Plot Harmony Clusters
# Need to fix funct to deal with harmony and Non resolutions
#fig = create_umap_visualisations(adata, 0.3, leiden_prefix="leiden_harmony", clustering_algorithm="Leiden")
#plt.show() 
sc.pl.umap(adata, color=['leiden_harmony_0.1'], ncols = 3) 
adata.obsm

In [ ]:

# Need to add sex, pcw info etc.
# obs_columns = ['sublibrary', 'plate']
plot_umap_grid(adata, obs_columns, grid_size = (2, 2), figsize = (10, 8), save_path = None)

### Percentage of cells per sample per cluster

In [ ]:
# Default
plot_stacked_figure(adata, sample_column = "sample", color_column = "leiden_0.1", barplot = True, recalculate_columns = True)

In [ ]:
# Harmony
plot_stacked_figure(adata, sample_column = "sample", color_column = "leiden_harmony_0.1", barplot = True, recalculate_columns = True)

In [ ]:
# Plot sample per cluster
# Function saves an excel file with the cell counts per sample per cluster 
# Extract sample and leiden cluster information from the AnnData object
fig = plot_and_save_cluster_percentages(
    adata = adata,
    output_dir = scanpy_dir,
    clustering_param = "leiden_harmony_0.1"
)


### Extract PCs as a DataFrame - need this for eQTL analysis expression covariates

In [ ]:

expression_pcs = pd.DataFrame(adata.obsm['X_pca'], index = adata.obs.index, 
                              columns = [f'PC{i+1}' for i in range(n_pcs)])
expression_pcs.to_csv(scanpy_dir + "expression_pcs.csv")

### Final counts

In [ ]:
adata.obs[['total_counts', 'n_counts', 'n_genes']]

### Save

In [ ]:
logger.info("Saving h5ad file ...")
adata.write(scanpy_dir + f'adata_clusters.h5ad')
logger.info("All done.")